# 第 22 课 · GQA、MQA 与紧凑 KV cache

**把“少存几组 K/V”这件事，从公式做到可测试的模型。**

前置知识：第16课多头注意力，第19课缓存，第20课 RoPE，第21课 RMSNorm/SwiGLU。

本课交付一份可独立执行的讲义。无需其他课程源码、模型下载或外部数据；只用 PyTorch 和标准库。CPU 可以完成，CUDA 是可选项。配套 `.py` 实现与这里一致；不要同时运行两份训练。

学习目标：解释 Hq/Hkv/d 三个量；写出正确的头分组；实现两套等价计算并核对梯度；紧凑保存缓存；完成 MHA→GQA/MQA 转换并解释为什么一般不等价。

**这不是无损压缩、量化或 MoE。** 默认实验固定人工语料与一个种子，不做架构排名或速度承诺。

## 0. 使用方法与可重复运行规则

按顺序执行；每个“先预测”问题先自己回答。独立练习在后半部分，参考答案不会覆盖你的函数。

已有 PyTorch 环境可以直接使用。缺少 Jupyter 时，在终端安装 `python -m pip install jupyterlab ipykernel`；GPU 版 PyTorch 沿用已有环境，不在讲义中自动安装或升级。

下方环境单元设置 `DEVICE_NAME="cpu"`，可改成 `"auto"`。首次运行默认训练一个小 MHA，然后转换三路继续训练。已有第21课 `rms_swiglu.pt` 时可指定 `SOURCE_CHECKPOINT`，跳过源模型训练。

生成文件写入**当前工作目录**下 `lesson_22_outputs`，同名文件覆盖。输入检查点与输出路径冲突时会报错。模型检查点只供恢复/生成，不是精确断点续训状态。

标为“复用”的长代码单元包含以前学过的实现，保证本课独立运行；本课优先精读第3、4、7节。

In [1]:
# -*- coding: utf-8 -*-
"""第22课：GQA、MQA 与紧凑 KV cache。

python lesson_22_gqa_mqa_and_compact_kv_cache.py --checks-only
python lesson_22_gqa_mqa_and_compact_kv_cache.py
python lesson_22_gqa_mqa_and_compact_kv_cache.py --checkpoint lesson_21_outputs/rms_swiglu.pt

仅依赖 PyTorch/标准库；导入不训练、不写文件、不联网。
主实现用显式分组维度，不在 Python 中复制 K/V 到 Hq 个头。
对照实现使用 repeat_interleave；两者前向和梯度应相同。
缓存只保存 [B,Hkv,T,d] 的旋转后 K 与未旋转 V。
训练支持右 padding；缓存仅 eval+no_grad、无 padding、同长批次。
完整模型仅 FP32/FP64；没有 AMP、分页缓存、量化或生产级性能保证。
保持第21课注意力投影偏置以支持严格加载，不能照搬为所有主流模型的配置。
MHA->GQA/MQA 的均值池化是有损结构转换；不是无损压缩，不承诺续训恢复质量。
默认 20 轮 MHA 预训练，再三路各续训 10 轮；小模板/单种子，不做架构排行榜。
生成/缓存比较针对同一模型，不要求不同架构输出相等。
只加载可信检查点；检查点保存权重/配置/词表，不是精确断点续训状态。

来源（正文讲义有具体引用）：
https://arxiv.org/html/2305.13245v3
https://arxiv.org/abs/1911.02150
https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html
https://github.com/meta-llama/llama3/blob/main/llama/model.py
"""
from __future__ import annotations
import argparse
from dataclasses import asdict, dataclass, replace
import hashlib
import json
import math
from pathlib import Path
import platform
import random
import tempfile
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100
CHECKPOINT_FAMILY = "lesson22_gqa_v1"

def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")

def nonnegative_int(value: int, name: str) -> None:
    if type(value) is not int or value < 0:
        raise ValueError(f"{name} 必须是非负整数，不接受布尔值")

def check_eps(eps: float) -> None:
    if (isinstance(eps, bool) or not isinstance(eps, (int, float))
            or not math.isfinite(eps) or eps <= 0):
        raise ValueError("eps 必须是正的有限数")

def check_base(base: float) -> None:
    if (isinstance(base, bool) or not isinstance(base, (int, float))
            or not math.isfinite(base) or base <= 1):
        raise ValueError("本课 rope_base 必须是大于 1 的有限数")

def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)

def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [2]:
DEVICE_NAME = "cpu"  # "auto" 表示 CUDA 可用就用，否则 CPU
DEVICE = choose_device(DEVICE_NAME)
torch.set_num_threads(1)
SEED = 22
SOURCE_CHECKPOINT = None  # 例如 Path("lesson_21_outputs/rms_swiglu.pt")
OUTPUT_DIR = Path("lesson_22_outputs")
EPOCHS = 20
ADAPT_EPOCHS = 10  # 设为0，只观察转换后结果；不进行额外训练
print("Python:", platform.python_version(), "PyTorch:", torch.__version__, "设备:", DEVICE)
print("输出目录:", OUTPUT_DIR.resolve())

Python: 3.13.5 PyTorch: 2.10.0+cpu 设备: cpu
输出目录: /mnt/data/qa22/notebook_run/lesson_22_outputs


## 1. 三种注意力，其实只改变了谁共享 K/V

本课把 Query 头数记为 $H_q$，K/V 头数记为 $H_{kv}$，每个头的维度记为 $d$。固定总隐藏维度 $D=H_qd$。

| 架构 | Query 头 | K/V 头 | 头的关系 |
|---|---:|---:|---|
| MHA | 4 | 4 | 每个 Query 头有自己的 K/V |
| GQA | 4 | 2 | 每两个 Query 头共享一组 K/V |
| MQA | 4 | 1 | 所有 Query 头共享同一组 K/V |

MQA 仍有**多个 Query 头**，不是单头注意力。广义 GQA 的两个端点分别是 MHA 和 MQA。[1][2]

本课均匀分组要求 $H_q\bmod H_{kv}=0$。变化的是 K/V 投影的输出宽度，而不是序列被切成几段；每个 Query 头仍可读取因果规则允许的全部历史位置。

先预测：`D=48,Hq=4,Hkv=2` 时，`d` 是12还是24？答案是 **12**，因为每头维度由 Query 头划分决定。

In [3]:
@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 48
    n_heads: int = 4  # Query 头数 Hq（保留旧名称以兼容第21课）
    n_kv_heads: int = 4  # K 和 V 头数相同，记作 Hkv
    n_layers: int = 2
    ff_dim: int = 192
    swiglu_dim: int = 128
    norm_kind: str = "rms"
    ffn_kind: str = "swiglu"
    norm_eps: float = 1e-6
    dropout: float = 0.0
    rope_base: float = 10000.0

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_kv_heads", "n_layers", "ff_dim", "swiglu_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= 4:
            raise ValueError("词表必须包含普通字符及四个特殊 token")
        if self.d_model % self.n_heads or (self.d_model // self.n_heads) % 2:
            raise ValueError("D 必须能被 H 整除，且每头维度为偶数")
        if self.n_heads % self.n_kv_heads:
            raise ValueError("Query 头数必须能被 KV 头数整除，且 Hkv <= Hq")
        if self.norm_kind not in {"layer", "rms"} or self.ffn_kind not in {"gelu", "swiglu"}:
            raise ValueError("不支持的归一化或 FFN 类型")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")
        check_eps(self.norm_eps)
        check_base(self.rope_base)

### 1.1 沿用的组件：本课不是从头重写所有代码

接下来两个单元复用 RMSNorm、SwiGLU、相邻配对 RoPE 和因果 mask。RoPE 仍分别处理 Q 和 K；二者的头数不同不影响每个头内的旋转。

为了严格兼容第21课，本课保留 Q/K/V/O 投影的偏置；不要由此推断所有实际模型都使用偏置。Dropout 默认为0，便于控制变量。

完整前向支持右侧 padding；缓存接口只接受 `eval()+no_grad()`、等长且无 PAD 的批次。修改权重或设备后必须丢弃旧缓存。

In [4]:
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """带缩放参数、不带偏置；FP16/BF16 先升 FP32，FP64 保留 FP64。

    输出 dtype 与 x 一致；这是本教学实现的明确精度约定。
    不减均值，不跨 batch/token 计算统计量，不在原地修改输入。
    """
    check_eps(eps)
    supported = (torch.float16, torch.bfloat16, torch.float32, torch.float64)
    if (not isinstance(x, torch.Tensor) or x.dtype not in supported
            or x.ndim < 1 or min(x.shape) <= 0):
        raise ValueError("x 必须是非空浮点张量，至少有一个维度")
    if (not isinstance(weight, torch.Tensor) or weight.dtype not in supported
            or weight.shape != (x.shape[-1],) or weight.device != x.device):
        raise ValueError("weight 必须是同设备、长度为最后特征维度的浮点向量")
    compute_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
    work = x.to(compute_dtype)
    mean_square = work.square().mean(dim=-1, keepdim=True)
    normalized = work * torch.rsqrt(mean_square + eps)
    return (normalized * weight.to(compute_dtype)).to(x.dtype)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        positive_int(dim, "dim")
        check_eps(eps)
        self.dim, self.eps = dim, float(eps)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return rms_norm(x, self.weight, self.eps)

def make_norm(kind: str, dim: int, eps: float) -> nn.Module:
    if kind == "rms":
        return RMSNorm(dim, eps)
    if kind == "layer":
        # LayerNorm 基线保留 gamma 和 beta；RMSNorm 仅有 gamma。
        return nn.LayerNorm(dim, eps=eps)
    raise ValueError("norm_kind 必须是 layer 或 rms")

class GELUFFN(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        positive_int(dim, "dim")
        positive_int(hidden, "hidden")
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.gelu(self.up_proj(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        positive_int(dim, "dim")
        positive_int(hidden, "hidden")
        self.gate_proj = nn.Linear(dim, hidden, bias=False)
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        value = self.up_proj(x)
        return self.down_proj(gate * value)

def make_ffn(cfg: ModelConfig) -> nn.Module:
    if cfg.ffn_kind == "gelu":
        return GELUFFN(cfg.d_model, cfg.ff_dim)
    return SwiGLUFFN(cfg.d_model, cfg.swiglu_dim)

In [5]:
def rope_frequencies(head_dim: int, base: float = 10000.0, *,
                     device: torch.device | str = "cpu", dtype=torch.float32) -> torch.Tensor:
    """omega_i=base**(-2*i/head_dim)，i 从 0 开始；不是 head_dim**(-2*i/base)。"""
    positive_int(head_dim, "head_dim")
    if head_dim % 2:
        raise ValueError("head_dim 必须是偶数")
    check_base(base)
    if dtype not in (torch.float32, torch.float64):
        raise ValueError("频率和相位使用 float32 或 float64，不直接使用半精度")
    exponent = torch.arange(0, head_dim, 2, device=device, dtype=dtype) / head_dim
    return float(base) ** (-exponent)

def _rope_inputs(x: torch.Tensor, positions: torch.Tensor, base: float) -> torch.dtype:
    if (not isinstance(x, torch.Tensor) or x.ndim != 4 or min(x.shape) <= 0
            or x.dtype not in (torch.float16, torch.bfloat16, torch.float32, torch.float64)
            or x.shape[-1] % 2):
        raise ValueError("x 必须是非空浮点 [B,H,T,d]，且 d 是偶数")
    if (not isinstance(positions, torch.Tensor) or positions.dtype != torch.long
            or positions.ndim != 1 or positions.shape[0] != x.shape[-2]
            or positions.device != x.device):
        raise ValueError("positions 必须是同设备、长度为 T 的 torch.long 向量")
    check_base(base)
    # 允许负位置：用于数学逆旋转实验；模型接口另外要求 offset 非负。
    return torch.float64 if x.dtype == torch.float64 else torch.float32

def apply_rope(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """相邻配对 (0,1),(2,3),...；返回相同 shape/dtype/device；不原地修改 x。

    输入 [B,H,T,d]，位置 [T]：本课各 batch 共享同一位置序列。
    半精度输入先升为 FP32 计算相位和旋转，再转回；主模型不启用 autocast。
    """
    compute_dtype = _rope_inputs(x, positions, base)
    freq = rope_frequencies(x.shape[-1], base, device=x.device, dtype=compute_dtype)
    angles = positions.to(compute_dtype)[:, None] * freq[None, :]
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    work = x.to(compute_dtype)
    even, odd = work[..., 0::2], work[..., 1::2]
    rotated_even = even*cos - odd*sin
    rotated_odd = even*sin + odd*cos
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2).to(x.dtype)

def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]

def offset_causal_mask(past_len: int, new_len: int, device: torch.device | str = 'cpu') -> torch.Tensor:
    """返回 [1,1,Q,P+Q]。第 i 个新 query 的绝对位置是 P+i。True=允许。"""
    if type(past_len) is not int or past_len < 0:
        raise ValueError('past_len 必须是非负整数')
    positive_int(new_len, 'new_len')
    query_positions = past_len + torch.arange(new_len, device=device)
    key_positions = torch.arange(past_len + new_len, device=device)
    allowed = key_positions[None, :] <= query_positions[:, None]
    return allowed[None, None, :, :]

## 2. 先把头的编号对应关系写对

令组大小 $G=H_q/H_{kv}$。编号为 $h$ 的 Query 头使用的 KV 头为：

$$g(h)=\left\lfloor h/G\right\rfloor.$$

例如 `Hq=8,Hkv=2`：

```text
Q头：  0 1 2 3 | 4 5 6 7
KV头： 0 0 0 0 | 1 1 1 1
```

这个映射是本课的**连续分组约定**。不能一处使用连续分组，另一处使用交错分组。

缓存仍按层保存，但其中张量的第二条轴变为 `Hkv`，而不是 `Hq`。[4][5]

In [6]:
LayerKV = tuple[torch.Tensor, torch.Tensor]

@dataclass(frozen=True)
class RotaryCache:
    layers: tuple[LayerKV, ...]
    position_offset: int
    rope_base: float
    owner: int  # 仅在当前进程识别模型对象；不是权重版本号，不是安全凭证。

    @property
    def length(self) -> int:
        return self.layers[0][0].shape[-2]

def cache_payload_bytes(cache: RotaryCache | None) -> int:
    if cache is None:
        return 0
    return sum(t.numel()*t.element_size() for pair in cache.layers for t in pair)

def kv_payload_estimate(layers: int, batch: int, length: int, kv_heads: int,
                        head_dim: int, element_bytes: int) -> int:
    """仅 K/V 有效元素的字节数，不是 GPU 峰值显存。"""
    for name, value in (("layers",layers),("batch",batch),("kv_heads",kv_heads),
                        ("head_dim",head_dim),("element_bytes",element_bytes)):
        positive_int(value,name)
    nonnegative_int(length,"length")
    return 2 * layers * batch * length * kv_heads * head_dim * element_bytes


def head_mapping(n_query_heads: int, n_kv_heads: int) -> list[int]:
    positive_int(n_query_heads,"n_query_heads")
    positive_int(n_kv_heads,"n_kv_heads")
    if n_query_heads % n_kv_heads:
        raise ValueError("Hq 必须能被 Hkv 整除")
    group = n_query_heads // n_kv_heads
    return [h // group for h in range(n_query_heads)]

In [7]:
for name, hkv in [("MHA",4),("GQA",2),("MQA",1)]:
    print(name, "Hq=4, Hkv=", hkv, "映射:", head_mapping(4,hkv))
print("Hq=8,Hkv=2:",head_mapping(8,2))

MHA Hq=4, Hkv= 4 映射: [0, 1, 2, 3]
GQA Hq=4, Hkv= 2 映射: [0, 0, 1, 1]
MQA Hq=4, Hkv= 1 映射: [0, 0, 0, 0]
Hq=8,Hkv=2: [0, 0, 0, 0, 1, 1, 1, 1]


## 3. 两种实现同一个 GQA 运算

### 版本一：重复 K/V，得到熟悉的四维注意力

把 K/V 的每个头连续重复 $G$ 次，形状从 `[B,Hkv,S,d]` 变成 `[B,Hq,S,d]`，再使用普通多头注意力公式。[3][4]

### 版本二：显式保留组维度

把 Q 拆为 `[B,Hkv,G,T,d]`；K/V 变为 `[B,Hkv,1,S,d]` 用于广播。

$$S_{b,g,r,i,j}=\frac{Q_{b,g,r,i,:}\cdot K_{b,g,j,:}}{\sqrt d}$$

沿 key 位置轴做 softmax，再与同组 V 相乘，最后合并 `Hkv,G` 为 `Hq`。

**本课主模型使用版本二。** 这是避免在 Python 层显式重复 K/V 的清晰写法；不保证底层广播矩阵乘没有临时复制，也不是融合 GPU 内核。注意力分数的总头数仍然是 `Hq`。

下面核心函数返回：`heads=[B,Hq,T,d]`，`weights=[B,Hq,T,S]`。权重是 Dropout 前的值。

In [8]:
def _attention_inputs(q, k, v, allowed):
    """本课核心输入约定：Q=[B,Hq,T,d]，K/V=[B,Hkv,S,d]。"""
    for name, value in (("q",q),("k",k),("v",v)):
        if (not isinstance(value,torch.Tensor) or value.ndim != 4
                or min(value.shape) <= 0 or not value.is_floating_point()):
            raise ValueError(f"{name} 必须是非空四维浮点张量")
    if k.shape != v.shape or q.shape[0] != k.shape[0] or q.shape[-1] != k.shape[-1]:
        raise ValueError("K/V 必须同形状，Q 与 K/V 的 batch、每头维度必须一致")
    if len({q.device,k.device,v.device}) != 1 or len({q.dtype,k.dtype,v.dtype}) != 1:
        raise ValueError("Q/K/V 必须具有相同的 device 和 dtype")
    head_mapping(q.shape[1],k.shape[1])
    if (not isinstance(allowed,torch.Tensor) or allowed.dtype != torch.bool
            or allowed.device != q.device or allowed.ndim != 4
            or allowed.shape[0] not in (1,q.shape[0]) or allowed.shape[1] != 1
            or allowed.shape[-2:] != (q.shape[-2],k.shape[-2])):
        raise ValueError("allowed 必须是同设备的 bool [B或1,1,T,S]，True=允许")
    if not bool(allowed.any(dim=-1).all()):
        raise ValueError("不允许某条 query 完全被屏蔽：普通 softmax 会出现 NaN")


def repeat_kv(x: torch.Tensor, n_query_heads: int) -> torch.Tensor:
    """只作为直观参考；重复结果不写入持久缓存。不是 Tensor.repeat。"""
    if not isinstance(x,torch.Tensor) or x.ndim != 4 or min(x.shape) <= 0:
        raise ValueError("KV 必须是非空 [B,Hkv,S,d] 张量")
    head_mapping(n_query_heads,x.shape[1])
    return x.repeat_interleave(n_query_heads // x.shape[1], dim=1)


def grouped_attention(q, k, v, allowed, *, dropout_p: float = 0.0,
                      training: bool = False) -> tuple[torch.Tensor,torch.Tensor]:
    """不显式 repeat K/V 的教学实现；返回输出和 dropout 前权重。

    把 Hq 分成 Hkv*G：Q=[B,Hkv,G,T,d]，每组共享 K/V。
    仍会生成每个 Query 头的分数矩阵，不是线性注意力或生产内核。
    底层 matmul 是否复制临时数据取决于实现，不承诺峰值内存。
    """
    _attention_inputs(q,k,v,allowed)
    if (type(training) is not bool or isinstance(dropout_p,bool)
            or not isinstance(dropout_p,(int,float)) or not 0 <= dropout_p < 1):
        raise ValueError("training 必须是 bool，dropout_p 必须在 [0,1)")
    b,hq,t,d = q.shape
    hkv,s = k.shape[1],k.shape[-2]
    group = hq // hkv
    q_grouped = q.reshape(b,hkv,group,t,d)
    scores = q_grouped @ k.unsqueeze(2).transpose(-2,-1) / math.sqrt(d)
    weights = torch.softmax(scores.masked_fill(~allowed.unsqueeze(2),float('-inf')),dim=-1)
    used = F.dropout(weights,p=dropout_p,training=training)
    heads = used @ v.unsqueeze(2)
    return heads.reshape(b,hq,t,d), weights.reshape(b,hq,t,s)


def attention_repeat_reference(q,k,v,allowed):
    """显式重复 K/V；独立的逐头四维实现，用于核对。"""
    _attention_inputs(q,k,v,allowed)
    repeated_k, repeated_v = repeat_kv(k,q.shape[1]),repeat_kv(v,q.shape[1])
    scores = q @ repeated_k.transpose(-2,-1) / math.sqrt(q.shape[-1])
    weights = torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
    return weights @ repeated_v,weights


def native_gqa_probe(device: torch.device) -> dict:
    """只探测本环境可用性；不把 enable_gqa=True 当作融合内核或省显存的证明。"""
    generator = torch.Generator(device=device).manual_seed(2207)
    q=torch.randn(1,4,2,4,device=device,dtype=torch.float64,generator=generator,requires_grad=True)
    k=torch.randn(1,2,5,4,device=device,dtype=torch.float64,generator=generator,requires_grad=True)
    v=torch.randn(1,2,5,4,device=device,dtype=torch.float64,generator=generator,requires_grad=True)
    mask=offset_causal_mask(3,2,device)
    expected,_=grouped_attention(q,k,v,mask)
    try:
        actual=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
                                             dropout_p=0.0,is_causal=False,enable_gqa=True)
        grads=torch.autograd.grad(actual.square().sum(),(q,k,v),retain_graph=True)
    except (TypeError,RuntimeError,NotImplementedError) as error:
        return {"status":"unsupported","reason":str(error),"device":str(device)}
    torch.testing.assert_close(actual,expected,atol=1e-10,rtol=1e-9)
    expected_grads=torch.autograd.grad(expected.square().sum(),(q,k,v))
    for actual_grad,expected_grad in zip(grads,expected_grads):
        torch.testing.assert_close(actual_grad,expected_grad,atol=1e-10,rtol=1e-9)
    return {"status":"passed","device":str(device),"dtype":"float64",
            "max_output_error":float((actual-expected).detach().abs().max()),
            "max_gradient_error":max(float((a-b).detach().abs().max()) for a,b in zip(grads,expected_grads)),
            "note":"只验证当前环境前向和梯度；未断言 FlashAttention 或显存收益"}

In [9]:
# 先预测 repeat 与 repeat_interleave 的顺序。
tags=torch.tensor([10.,20.]).reshape(1,2,1,1)
print("repeat_interleave:",repeat_kv(tags,4).flatten().tolist())
print("普通 repeat:",tags.repeat(1,2,1,1).flatten().tolist())
print("只有第一种符合本课连续分组约定。")

repeat_interleave: [10.0, 10.0, 20.0, 20.0]
普通 repeat: [10.0, 20.0, 10.0, 20.0]
只有第一种符合本课连续分组约定。


In [10]:
# 方阵之外再看矩形：旧历史2个位置，本次新增3个位置。
set_seed(2202)
q=torch.randn(2,4,3,6,dtype=torch.float64)
k=torch.randn(2,2,5,6,dtype=torch.float64)
v=torch.randn_like(k)
allowed=offset_causal_mask(2,3)
heads,weights=grouped_attention(q,k,v,allowed)
ref,ref_w=attention_repeat_reference(q,k,v,allowed)
print("Q:",tuple(q.shape),"K/V:",tuple(k.shape))
print("输出:",tuple(heads.shape),"权重:",tuple(weights.shape))
print("允许矩阵:\n",allowed[0,0].int())
print("两个实现的最大误差:",(heads-ref).abs().max().item())
torch.testing.assert_close(heads,ref)
torch.testing.assert_close(weights.sum(-1),torch.ones_like(weights.sum(-1)))

Q: (2, 4, 3, 6) K/V: (2, 2, 5, 6)
输出: (2, 4, 3, 6) 权重: (2, 4, 3, 5)
允许矩阵:
 tensor([[1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)
两个实现的最大误差: 0.0


### 3.1 共享 K/V 不等于共享注意力权重

同组的 Query 向量仍然不同。即使看相同的 K/V，它们也可以给各个位置分配不同权重。

下一个例子只有一组 K/V，但两个 Query 分别偏向第一个与第二个坐标。运行前预测权重是否一样。

In [11]:
q=torch.tensor([[[[2.,0.]],[[0.,2.]]]])  # 两个 Query 头
k=torch.eye(2)[None,None]               # 一组 K
v=k.clone()                            # 一组 V
heads,weights=grouped_attention(q,k,v,torch.ones(1,1,1,2,dtype=torch.bool))
print("每个 Query 头的权重:\n",weights[0,:,0,:])
print("每个 Query 头的输出:\n",heads[0,:,0,:])
assert not torch.allclose(weights[:,0],weights[:,1])

每个 Query 头的权重:
 tensor([[0.8044, 0.1956],
        [0.1956, 0.8044]])
每个 Query 头的输出:
 tensor([[0.8044, 0.1956],
        [0.1956, 0.8044]])


### 3.2 共享参数还意味着：梯度来自所有使用它的 Query 头

不能在重复 K/V 时偷偷 `detach()`。下例把一个 KV 头重复4次，所有元素求和后，它收到的梯度应当是4，而不是1或None。

完整检查还会对比分组实现与重复实现的 Q/K/V 梯度，覆盖 MHA、GQA、MQA 三个端点。

In [12]:
shared=torch.tensor([[[[1.,2.]]]],requires_grad=True)
repeat_kv(shared,4).sum().backward()
print("共享 KV 的梯度:",shared.grad)
torch.testing.assert_close(shared.grad,torch.full_like(shared,4))
print("原生 enable_gqa 探测:",native_gqa_probe(DEVICE))

共享 KV 的梯度: tensor([[[[4., 4.]]]])
原生 enable_gqa 探测: {'status': 'passed', 'device': 'cpu', 'dtype': 'float64', 'max_output_error': 3.3306690738754696e-16, 'max_gradient_error': 9.43689570931383e-16, 'note': '只验证当前环境前向和梯度；未断言 FlashAttention 或显存收益'}


**原生接口的限制。** `F.scaled_dot_product_attention(...,enable_gqa=True)` 能表达这种分组，但实际后端、设备、dtype 和版本支持需要核实；调用成功不代表使用了某个融合内核。官方文档还要求 Query 头数能被 KV 头数整除，K 与 V 头数相同。[3]

本课不依赖原生 GQA 才能训练；它只是额外核对。不可用时明确记为 `unsupported`，不伪装成通过。

有缓存的矩形注意力仍使用偏移 mask，传给 SDPA 时使用 `is_causal=False`，避免直接套用左上对齐的方阵习惯。

## 4. 投影、RoPE、紧凑缓存和多层模型

固定 `D=48,Hq=4,d=12`，GQA 使用 `Hkv=2`：

| 步骤 | 形状 |
|---|---|
| 输入 | `[B,T,48]` |
| Q 投影与拆头 | `[B,4,T,12]` |
| K/V 投影与拆头 | `[B,2,T,12]` |
| 注意力权重 | `[B,4,T,S]` |
| 各 Query 头输出 | `[B,4,T,12]` |
| 合并并输出投影 | `[B,T,48]` |

**不能再用一次 `chunk(3)` 假设 Q/K/V 总宽度相等。** 本课分别定义三次 Linear。

新增 K 先做 RoPE，再按位置轴追加；旧 K 不能重复旋转。缓存中始终保留紧凑头数 `Hkv`，不能把参考实现的重复结果存回缓存。

In [13]:
class RotaryAttention(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.n_heads, self.d_head = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.base = cfg.rope_base
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_head)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_head)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def project(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        b, t, d = x.shape
        def split(proj, heads):
            return proj(x).reshape(b, t, heads, self.d_head).transpose(1, 2)
        return (split(self.q_proj, self.n_heads), split(self.k_proj, self.n_kv_heads),
                split(self.v_proj, self.n_kv_heads))

    def combine(self, q, k, v, allowed):
        heads, _ = grouped_attention(q, k, v, allowed,
                                     dropout_p=self.dropout.p, training=self.training)
        b, h, t, d = heads.shape
        return self.out_proj(heads.transpose(1, 2).contiguous().reshape(b, t, h*d))

    def forward(self, x: torch.Tensor, positions: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        q, k, v = self.project(x)
        q = apply_rope(q, positions, self.base)
        k = apply_rope(k, positions, self.base)
        return self.combine(q, k, v, allowed)  # V 不旋转。

    def forward_cached(self, x: torch.Tensor, positions: torch.Tensor,
                       past: LayerKV | None = None) -> tuple[torch.Tensor, LayerKV]:
        q, k_new, v_new = self.project(x)
        q = apply_rope(q, positions, self.base)
        k_new = apply_rope(k_new, positions, self.base)
        past_len = 0 if past is None else past[0].shape[-2]
        if past is None:
            k_all, v_all = k_new, v_new
        else:
            # past[0] 已按各自原位置旋转，不能对拼接后的 k_all 再旋转！
            k_all = torch.cat((past[0], k_new), dim=-2)
            v_all = torch.cat((past[1], v_new), dim=-2)
        allowed = offset_causal_mask(past_len, x.shape[1], x.device)
        return self.combine(q, k_all, v_all, allowed), (k_all, v_all)

In [14]:
class ModernBlock(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.norm1 = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.attn = RotaryAttention(cfg)
        self.norm2 = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.ffn = make_ffn(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, positions, allowed):
        x = x + self.dropout(self.attn(self.norm1(x), positions, allowed))
        return x + self.dropout(self.ffn(self.norm2(x)))

class ModernGPT(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([ModernBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def _input_check(self, ids):
        if (not isinstance(ids, torch.Tensor) or ids.ndim != 2 or min(ids.shape) <= 0
                or ids.dtype != torch.long):
            raise ValueError("输入必须是非空 [B,T] torch.long")
        if ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须在同一设备")
        if bool(((ids < 0) | (ids >= self.cfg.vocab_size)).any()):
            raise ValueError("token 编号越界")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过本模型的教学上下文上限；不自动截断")
        if self.token_embedding.weight.dtype not in (torch.float32, torch.float64):
            raise ValueError("完整教学模型仅支持 FP32/FP64；半精度只用于独立组件实验")
        if torch.is_autocast_enabled(ids.device.type):
            raise ValueError("本课未实现 autocast 路径")

    def forward(self, input_ids: torch.Tensor, *, position_offset: int = 0) -> torch.Tensor:
        self._input_check(input_ids)
        nonnegative_int(position_offset, "position_offset")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = position_offset + torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        for block in self.blocks:
            x = block(x, positions, allowed)
        return self.lm_head(self.final_norm(x))

    def _cache_check(self, ids, cache, offset):
        self._input_check(ids)
        if any(m.training for m in self.modules()) or torch.is_grad_enabled():
            raise ValueError("缓存路径只接受 eval + torch.no_grad() 推理")
        if bool((ids == PAD_ID).any()):
            raise ValueError("缓存路径不接受 padding；不同长度提示请分开处理")
        if offset is not None:
            nonnegative_int(offset, "position_offset")
        if cache is None:
            return 0, (0 if offset is None else offset)
        if (not isinstance(cache, RotaryCache) or not isinstance(cache.layers, tuple)
                or len(cache.layers) != self.cfg.n_layers):
            raise ValueError("缓存必须是 RotaryCache，且层数与模型一致")
        if cache.owner != id(self) or cache.rope_base != self.cfg.rope_base:
            raise ValueError("缓存所属模型或 RoPE 配置不一致，请重新 prefill")
        nonnegative_int(cache.position_offset, "cache.position_offset")
        if offset is not None and offset != cache.position_offset:
            raise ValueError("不能中途改变缓存的起始位置；请重新 prefill")
        lengths = []
        for pair in cache.layers:
            if not isinstance(pair, tuple) or len(pair) != 2:
                raise ValueError("每层缓存必须是 (K,V)")
            k, v = pair
            if (not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor)
                    or k.ndim != 4 or k.shape != v.shape or k.shape[0] != ids.shape[0]
                    or k.shape[1] != self.cfg.n_kv_heads or k.shape[-1] != self.cfg.d_model // self.cfg.n_heads
                    or k.shape[-2] <= 0):
                raise ValueError("缓存需要紧凑形状 [B,Hkv,P,d_head]，P>0")
            for t in pair:
                if (t.device != ids.device or t.dtype != self.token_embedding.weight.dtype
                        or t.requires_grad):
                    raise ValueError("缓存 device/dtype/求导状态不匹配")
            lengths.append(k.shape[-2])
        if len(set(lengths)) != 1:
            raise ValueError("各层历史长度必须一致")
        return lengths[0], cache.position_offset

    def forward_cached(self, input_ids: torch.Tensor, cache: RotaryCache | None = None, *,
                       position_offset: int | None = None) -> tuple[torch.Tensor, RotaryCache]:
        past_len, offset = self._cache_check(input_ids, cache, position_offset)
        new_len = input_ids.shape[1]
        if past_len + new_len > self.cfg.max_len:
            raise ValueError("历史加新增超过上下文；本课不滑动窗口")
        positions = offset + past_len + torch.arange(new_len, device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        updated = []
        for index, block in enumerate(self.blocks):
            past = None if cache is None else cache.layers[index]
            delta, kv = block.attn.forward_cached(block.norm1(x), positions, past)
            x = x + block.dropout(delta)
            x = x + block.dropout(block.ffn(block.norm2(x)))
            updated.append(kv)
        new_cache = RotaryCache(tuple(updated), offset, self.cfg.rope_base, id(self))
        return self.lm_head(self.final_norm(x)), new_cache

In [15]:
cfg_demo=ModelConfig(vocab_size=12,n_kv_heads=2)
attn_demo=RotaryAttention(cfg_demo)
q,k,v=attn_demo.project(torch.randn(2,5,48))
print("Q:",q.shape,"K:",k.shape,"V:",v.shape)
print("Q权重:",attn_demo.q_proj.weight.shape,"K权重:",attn_demo.k_proj.weight.shape)
print("KV总宽度 =",cfg_demo.n_kv_heads*(cfg_demo.d_model//cfg_demo.n_heads))

Q: torch.Size([2, 4, 5, 12]) K: torch.Size([2, 2, 5, 12]) V: torch.Size([2, 2, 5, 12])
Q权重: torch.Size([48, 48]) K权重: torch.Size([24, 48])
KV总宽度 = 24


## 5. 缓存省多少？先算载荷，再谈显存

K、V 两份状态，$L$层、batch $B$、长度 $T$、$H_{kv}$个头、每头 $d$个元素，每元素 $s$字节：

$$\text{KV bytes}=2LBTH_{kv}ds.$$

固定其他量时，GQA/MQA 相对 MHA 的缓存载荷比例是 $H_{kv}/H_q$。

下一个计算例子只是**假设配置**，不是本课训练模型、也不是3060容量保证。模型权重、注意力临时张量、其他激活、缓存分配器和框架开销都不包含在公式里。

结构节省和内核节省也不同：Query 头数未减少，显式注意力权重仍可能占据 `[B,Hq,T,S]` 的空间。因此不能从“KV 少4倍”推导“整体显存少4倍”或“速度快4倍”。[1][2]

In [16]:
# 假设：32层、batch=1、长度8192、Hq=32、每头128、每元素2字节。
for label, hkv in [("MHA",32),("GQA",8),("MQA",1)]:
    size=kv_payload_estimate(32,1,8192,hkv,128,2)
    print(f"{label}: Hkv={hkv}, 仅KV={size/(1024**3):.3f} GiB")

MHA: Hkv=32, 仅KV=4.000 GiB
GQA: Hkv=8, 仅KV=1.000 GiB
MQA: Hkv=1, 仅KV=0.125 GiB


### 5.1 注意力参数量也变了，但整网参数不会同比例减少

不含偏置时：

$$N_{attn}=2D^2+2D(H_{kv}d).$$

前半项来自 Q 和输出投影，后半项来自 K、V。即使 MQA 只有一个 KV 头，Q 和输出投影仍然存在。

本课为了兼容第21课，还保留了 $2D+2H_{kv}d$ 个偏置参数。Embedding、RMSNorm、FFN 和词表头没有因为减少 KV 头就同步缩小。

## 6. 从 MHA 检查点转换：为什么通常有损？

GQA 论文采用把每组原有 K/V 投影做均值池化，再继续预训练的路线。[1]

PyTorch Linear 权重是 `[out_features,in_features]`。原 K 权重 `[Hq*d,D]` 要先看作 `[Hkv,G,d,D]`，**沿 G 求平均**，不是沿 d 或 D 求平均。V 和偏置也需对应处理。

$$W^{K,new}_{g}=\frac1G\sum_{r=0}^{G-1}W^{K,old}_{gG+r}.$$

因为原来每个 Query 使用不同的 K/V，转换后变成共享平均值，加上 softmax 的非线性，不能期待一般输入输出保持不变。

特殊情况：原组内的 K/V 投影本来就完全一致，此时池化不会改变它们。讲义检查包含这个可验证的边界，而不是笼统声称“所有转换都有固定误差”。

In [17]:
def pool_kv_projection(tensor: torch.Tensor, n_query_heads: int,
                       n_kv_heads: int, head_dim: int) -> torch.Tensor:
    """Linear.weight=[out,in] 或 bias=[out]；对相邻头分组，沿组内头轴平均。

    不是对 head_dim 平均，也不是沿输入特征轴平均；保留所有内容坐标。
    """
    head_mapping(n_query_heads,n_kv_heads)
    positive_int(head_dim,"head_dim")
    if (not isinstance(tensor,torch.Tensor) or tensor.ndim not in (1,2)
            or not tensor.is_floating_point() or tensor.shape[0] != n_query_heads*head_dim):
        raise ValueError("需要 [Hq*d,D] 投影权重或 [Hq*d] 偏置")
    group=n_query_heads//n_kv_heads
    grouped=tensor.reshape(n_kv_heads,group,head_dim,*tensor.shape[1:])
    return grouped.mean(dim=1).reshape(n_kv_heads*head_dim,*tensor.shape[1:])


def convert_mha(source: ModernGPT, n_kv_heads: int) -> ModernGPT:
    """不修改 source。一般情况是有损结构转换，不能据此声称预测等价。

    Q/out/FFN/norm/embedding 原样复制，只对 K/V 投影权重和偏置分组平均。
    返回新模型且无缓存；禁止通过 strict=False 静默跳过不匹配权重。
    """
    if source.cfg.n_kv_heads != source.cfg.n_heads:
        raise ValueError("本课转换入口必须是 MHA：源 Hkv==Hq")
    head_mapping(source.cfg.n_heads,n_kv_heads)
    cfg=replace(source.cfg,n_kv_heads=n_kv_heads)
    parameter=next(source.parameters())
    with torch.random.fork_rng(devices=[]):
        converted=ModernGPT(cfg).to(device=parameter.device,dtype=parameter.dtype)
    destination={}
    for name,tensor in source.state_dict().items():
        if ".attn.k_proj." in name or ".attn.v_proj." in name:
            destination[name]=pool_kv_projection(tensor,cfg.n_heads,n_kv_heads,cfg.d_model//cfg.n_heads)
        else:
            destination[name]=tensor.clone()
    converted.load_state_dict(destination,strict=True)
    converted.train(source.training)
    return converted


def attention_parameter_count(cfg: ModelConfig) -> dict:
    d=cfg.d_model
    kv_width=cfg.n_kv_heads*(d//cfg.n_heads)
    weights=2*d*d+2*d*kv_width
    biases=2*d+2*kv_width  # 为严格兼容第21课而保留偏置。
    return {"Hq":cfg.n_heads,"Hkv":cfg.n_kv_heads,"d_head":d//cfg.n_heads,
            "weights":weights,"biases":biases,"total_per_layer":weights+biases}

In [18]:
w=torch.arange(32.,dtype=torch.float64).reshape(8,4)
print("原权重的4个头:\n",w.reshape(4,2,4))
print("合并为2个KV头:\n",pool_kv_projection(w,4,2,2).reshape(2,2,4))
for hkv in (4,2,1):
    cfg=ModelConfig(vocab_size=41,n_kv_heads=hkv)
    print("Hkv=",hkv,attention_parameter_count(cfg))

原权重的4个头:
 tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.]],

        [[ 8.,  9., 10., 11.],
         [12., 13., 14., 15.]],

        [[16., 17., 18., 19.],
         [20., 21., 22., 23.]],

        [[24., 25., 26., 27.],
         [28., 29., 30., 31.]]], dtype=torch.float64)
合并为2个KV头:
 tensor([[[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[20., 21., 22., 23.],
         [24., 25., 26., 27.]]], dtype=torch.float64)
Hkv= 4 {'Hq': 4, 'Hkv': 4, 'd_head': 12, 'weights': 9216, 'biases': 192, 'total_per_layer': 9408}
Hkv= 2 {'Hq': 4, 'Hkv': 2, 'd_head': 12, 'weights': 6912, 'biases': 144, 'total_per_layer': 7056}
Hkv= 1 {'Hq': 4, 'Hkv': 1, 'd_head': 12, 'weights': 5760, 'biases': 120, 'total_per_layer': 5880}


## 7. 准备完整实验：转换前、转换后、再训练后分开测

流程不是“训练三个毫无关系的模型再比较”：

1. 训练或加载同一个 MHA 源模型；源模型之后保持不变。
2. 复制一份保留 MHA；另两份按组均值转为 GQA、MQA。
3. 先立即测验证损失与 logits 变化。
4. 三路分别用相同样本顺序和相同更新次数继续训练，重新创建优化器。

**MHA继续训练这一路不能省略。** 否则可能把额外训练的收益误归因给结构转换。参数量和计算量仍然不同，这也不是严格等FLOPs比较。

默认源模型训练20轮，三路各续训10轮；选择验证损失最好的轮次，同时把转换后第0轮纳入候选。论文的额外预训练比例是它的特定实验设置，本课不声称复现原论文5%算力结论。[1]

下面复用之前的数据、损失、验证、生成函数。

In [19]:
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]

def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts

def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]

def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token

def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")

def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]

class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]

def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }

def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))

def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count

def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}

In [20]:
def evaluate(model: ModernGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)

def train_epoch(model: ModernGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。

def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

def model_fingerprint(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str((tuple(value.shape), value.dtype)).encode('utf-8'))
        digest.update(bytes(value.reshape(-1).view(torch.uint8).tolist()))
    return digest.hexdigest()

In [21]:
def generate_greedy(model: ModernGPT, vocab: list[str], prompt: str = '', *,
                    use_cache: bool = True, max_new_tokens: int = 24) -> dict:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size or not isinstance(prompt, str):
        raise ValueError('词表和模型不一致，或 prompt 不是字符串')
    if type(use_cache) is not bool or type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError('use_cache 必须是布尔值；max_new_tokens 必须是非负整数')
    ids = [BOS_ID] + encode(prompt, {t: i for i, t in enumerate(vocab)})
    if len(ids) > model.cfg.max_len:
        raise ValueError('提示超过上下文；不能静默截断')
    device = next(model.parameters()).device
    current = torch.tensor([ids], device=device, dtype=torch.long)
    cache, trace, new_ids = None, [], []
    reason = 'max_new_tokens' if max_new_tokens == 0 else 'context_limit'
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for step in range(max_new_tokens):
                # 与前课一致：本接口将返回的总长度限制在 max_len，不多吐出第 max_len+1 个编号。
                if current.shape[1] >= model.cfg.max_len:
                    reason = 'context_limit'
                    break
                fed = current if not use_cache or cache is None else current[:, -1:]
                if use_cache:
                    logits, cache = model.forward_cached(fed, cache)
                else:
                    logits = model(fed)
                last = logits[:, -1, :].clone()
                last[:, [BOS_ID, PAD_ID]] = float('-inf')
                next_id = int(last.argmax(-1).item())
                new_ids.append(next_id)
                trace.append({'prediction_step': step, 'input_tokens_this_call': fed.shape[1],
                              'processed_prefix_tokens': current.shape[1],
                              'cache_tokens_after_call': 0 if cache is None else cache.length,
                              'selected_token_id': next_id})
                current = torch.cat((current, current.new_tensor([[next_id]])), dim=1)
                if next_id == EOS_ID:
                    reason = 'eos'
                    break
                reason = 'max_new_tokens'
    finally:
        model.train(was_training)
    visible = ''.join(vocab[i] for i in new_ids if i not in (BOS_ID, EOS_ID, PAD_ID))
    return {'prompt': prompt, 'text': prompt + visible, 'use_cache': use_cache,
            'new_ids': new_ids, 'stop_reason': reason, 'trace': trace,
            'cache_tokens_at_return': 0 if cache is None else cache.length,
            'cache_payload_bytes': cache_payload_bytes(cache),
            'note': '最后选出的 token 尚未送回模型，因此通常未进入 cache；cache 不作为可续接状态返回。'}

def compare_chunked(model: ModernGPT, ids: torch.Tensor, chunks: list[int], *, offset: int = 0) -> dict:
    if not chunks or any(type(c) is not int or c <= 0 for c in chunks) or sum(chunks) != ids.shape[1]:
        raise ValueError("chunks 必须是正整数列表，总和等于 T")
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            full = model(ids, position_offset=offset)
            prefill, full_cache = model.forward_cached(ids, position_offset=offset)
            cache, outputs, start, trace = None, [], 0, []
            for size in chunks:
                logits, cache = model.forward_cached(ids[:,start:start+size], cache, position_offset=offset)
                outputs.append(logits)
                trace.append({"past":start, "new":size, "positions":list(range(offset+start,offset+start+size)),
                              "cache_length":cache.length, "K_shape":list(cache.layers[0][0].shape)})
                start += size
            chunked = torch.cat(outputs, dim=1)
            torch.testing.assert_close(prefill, full, rtol=3e-4, atol=3e-5)
            torch.testing.assert_close(chunked, full, rtol=3e-4, atol=3e-5)
            max_kv = 0.0
            for pair, expected_pair in zip(cache.layers, full_cache.layers):
                for value, expected in zip(pair, expected_pair):
                    torch.testing.assert_close(value, expected, rtol=3e-4, atol=3e-5)
                    max_kv = max(max_kv, (value-expected).abs().max().item())
            shifted = model(ids, position_offset=offset+31)
            torch.testing.assert_close(shifted, full, rtol=3e-4, atol=3e-5)
            return {"chunks":chunks, "position_offset":offset, "trace":trace,
                    "max_logits_error":(chunked-full).abs().max().item(),
                    "max_cache_error":max_kv, "global_shift_logits_error":(shifted-full).abs().max().item(),
                    "cache_bytes":cache_payload_bytes(cache), "rtol":3e-4, "atol":3e-5}
    finally:
        model.train(was_training)

In [22]:
def train_variant(model: ModernGPT, train_texts: list[str], val_texts: list[str],
                  vocab: list[str], device: torch.device, *, epochs: int, seed: int,
                  name: str, learning_rate: float = 0.003) -> dict:
    nonnegative_int(epochs, "epochs")
    mapping = {t: i for i, t in enumerate(vocab)}
    train_data = CharDataset(train_texts, mapping, model.cfg.max_len)
    val_data = CharDataset(val_texts, mapping, model.cfg.max_len)
    # 每个变体都使用新的同种子 loader，所以各轮样本顺序一致。
    train_loader = make_loader(train_data, 16, True, seed)
    train_eval = make_loader(train_data, 16)
    val_loader = make_loader(val_data, 16)
    set_seed(seed)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    before = {"train": evaluate(model, train_eval, device),
              "validation": evaluate(model, val_loader, device)}
    best_state = snapshot_state(model)
    best_epoch, best_loss = 0, before["validation"]["loss"]
    history = []
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        row = {"epoch": epoch, "online_train": online,
               "train": evaluate(model, train_eval, device),
               "validation": evaluate(model, val_loader, device)}
        history.append(row)
        if row["validation"]["loss"] < best_loss:
            best_loss, best_epoch = row["validation"]["loss"], epoch
            best_state = snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"{name:13s} epoch={epoch:02d} train={row['train']['loss']:.4f} val={row['validation']['loss']:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    model.zero_grad(set_to_none=True)
    return {"seed": seed, "epochs": epochs, "config": asdict(model.cfg),
            "parameters": count_parameters(model), "before": before, "history": history,
            "best_epoch": best_epoch,
            "selected_train": evaluate(model, train_eval, device),
            "selected_validation": evaluate(model, val_loader, device),
            "updates": sum(row["online_train"]["steps"] for row in history)}

### 7.1 检查点与输出约定

`load_checkpoint()` 对第21课结构增加 `n_kv_heads=n_heads`，然后严格加载，不使用 `strict=False` 跳过参数。`convert_mha()` 是另一个显式步骤。

源文件不会修改；目标模型的参数独立存储。缓存是请求状态，不随检查点保存，也不能从源 MHA 直接挪给新 GQA。

默认训练语料仍是96条人工模板，训练77条、验证19条；只用训练文本建词表。加载其他来源时，这份固定语料可能不对应源模型训练集，报告会标记检查点来源，不能把指标理解成严格留出泛化。

输出：`mha_source.pt`、`mha_continued.pt`、`gqa.pt`、`mqa.pt`、`metrics.json`、`training.json`、`split.json`。

In [23]:
def save_checkpoint(path: Path, model: ModernGPT, vocab: list[str], best_epoch: int) -> None:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size:
        raise ValueError("词表与模型不一致")
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    torch.save({"family":CHECKPOINT_FAMILY,"config":asdict(model.cfg),"vocab":vocab,
                "state_dict":snapshot_state(model),"best_epoch":best_epoch,
                "purpose":"inference_only_not_exact_training_resume"},path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[ModernGPT,list[str],dict]:
    """可读取本课检查点和第21课的严格兼容 MHA；仅加载自己生成或可信的文件。"""
    payload=torch.load(Path(path),map_location='cpu',weights_only=True)
    if not isinstance(payload,dict):
        raise ValueError("检查点必须是带配置和词表的字典")
    family=payload.get('family')
    if family not in {CHECKPOINT_FAMILY,'lesson21_rms_swiglu_v1'}:
        raise ValueError("只接受第21/22课检查点，不接受其他架构")
    config=dict(payload['config'])
    if family=='lesson21_rms_swiglu_v1':
        config['n_kv_heads']=config['n_heads']
    cfg=ModelConfig(**config)
    vocab=payload['vocab']
    validate_vocab(vocab)
    if cfg.vocab_size!=len(vocab):
        raise ValueError("词表大小不匹配")
    with torch.random.fork_rng(devices=[]):
        model=ModernGPT(cfg)
    model.load_state_dict(payload['state_dict'],strict=True)
    model.to(device).eval()
    return model,vocab,{"source_family":family,"best_epoch":payload.get('best_epoch')}


def resource_probe(model: ModernGPT, length: int = 32) -> dict:
    """用固定长度输入核对缓存形状/有效载荷；不把这些数字称为峰值显存。"""
    length=min(length,model.cfg.max_len)
    device=next(model.parameters()).device
    ids=torch.full((1,length),4,device=device,dtype=torch.long)
    ids[:,0]=BOS_ID
    was_training=model.training
    try:
        model.eval()
        with torch.no_grad():
            _,cache=model.forward_cached(ids)
        actual=cache_payload_bytes(cache)
        size=cache.layers[0][0].element_size()
        expected=kv_payload_estimate(model.cfg.n_layers,1,length,model.cfg.n_kv_heads,
                                     model.cfg.d_model//model.cfg.n_heads,size)
        if actual!=expected:
            raise AssertionError("缓存没有按 Hkv 紧凑保存")
        return {"length":length,"cache_shape_per_tensor":list(cache.layers[0][0].shape),
                "actual_cache_bytes":actual,"formula_cache_bytes":expected,
                "parameters":count_parameters(model),
                "attention_parameters":attention_parameter_count(model.cfg),
                "note":"只计算有效 K/V 元素，不是进程/GPU 峰值，不含临时张量"}
    finally:
        model.train(was_training)


def conversion_probe(source: ModernGPT, converted: ModernGPT, ids: torch.Tensor) -> dict:
    old_source,old_new=source.training,converted.training
    try:
        source.eval(); converted.eval()
        with torch.no_grad():
            original=source(ids)
            changed=converted(ids)
        return {"max_logits_change":float((original-changed).abs().max()),
                "same_argmax_fraction":float((original.argmax(-1)==changed.argmax(-1)).float().mean()),
                "note":"跨架构变化；不是完整前向与缓存之间的误差"}
    finally:
        source.train(old_source);converted.train(old_new)


def run_experiment(device: torch.device, output_dir: Path, *, epochs: int = 20,
                   adapt_epochs: int = 10, seed: int = 22, checkpoint: Path | None = None) -> dict:
    positive_int(epochs,'epochs');nonnegative_int(adapt_epochs,'adapt_epochs')
    output_dir=Path(output_dir)
    outputs=['mha_source.pt','mha_continued.pt','gqa.pt','mqa.pt','metrics.json','training.json','split.json']
    if checkpoint is not None and any(checkpoint.resolve()==(output_dir/f).resolve() for f in outputs):
        raise ValueError("输入检查点与输出文件重合，请指定不同的 --output-dir")
    train_texts,val_texts=split_texts(clean_texts(demo_lines()),seed=17)
    if checkpoint is None:
        _,vocab=build_vocab(train_texts)
        set_seed(seed)
        source=ModernGPT(ModelConfig(vocab_size=len(vocab))).to(device)
        source_training=train_variant(source,train_texts,val_texts,vocab,device,
                                      epochs=epochs,seed=seed,name='mha_source')
    else:
        source,vocab,info=load_checkpoint(checkpoint,device)
        if source.cfg.n_kv_heads!=source.cfg.n_heads:
            raise ValueError("--checkpoint 必须提供 MHA 源模型；不能把已转换 GQA/MQA 当作 MHA")
        if source.cfg.n_heads<4 or source.cfg.n_heads%2:
            raise ValueError("三路演示需要至少4个且为偶数的 Query 头")
        source_training={"source":"checkpoint","path":str(checkpoint),**info,
                         "note":"不假设源文件语料/种子与本课相同；续训用本课固定划分"}
    source.eval()
    before_fingerprint=model_fingerprint(source)
    source_metrics=evaluate(source,make_loader(CharDataset(val_texts,{t:i for i,t in enumerate(vocab)},source.cfg.max_len),16),device)
    output_dir.mkdir(parents=True,exist_ok=True)
    save_checkpoint(output_dir/'mha_source.pt',source,vocab,source_training.get('best_epoch',0))
    probe_ids=torch.tensor([[BOS_ID]+encode('我今天',{t:i for i,t in enumerate(vocab)})],device=device)
    targets={'mha_continued':source.cfg.n_heads,'gqa':source.cfg.n_heads//2,'mqa':1}
    rows,history={},{}
    for name,kv_heads in targets.items():
        model=convert_mha(source,kv_heads)
        immediate=conversion_probe(source,model,probe_ids)
        # 包括 MHA 继续训练对照；三路重置相同优化器/数据顺序，不宣称 FLOPs 相同。
        training=train_variant(model,train_texts,val_texts,vocab,device,
                               epochs=adapt_epochs,seed=seed+1,name=name,learning_rate=0.001)
        fingerprint=model_fingerprint(model)
        cache_check=compare_chunked(model,probe_ids,[1,1,probe_ids.shape[1]-2],offset=7)
        cached=generate_greedy(model,vocab,'我今天正在',use_cache=True,max_new_tokens=16)
        recomputed=generate_greedy(model,vocab,'我今天正在',use_cache=False,max_new_tokens=16)
        equal=(cached['new_ids']==recomputed['new_ids'] and cached['stop_reason']==recomputed['stop_reason'])
        save_checkpoint(output_dir/f'{name}.pt',model,vocab,training['best_epoch'])
        restored,restored_vocab,_=load_checkpoint(output_dir/f'{name}.pt',device)
        with torch.no_grad():
            reload_error=float((model(probe_ids)-restored(probe_ids)).abs().max())
        if reload_error!=0 or restored_vocab!=vocab or model_fingerprint(model)!=fingerprint:
            raise AssertionError('保存重载或推理未改变参数的检查失败')
        rows[name]={"Hq":model.cfg.n_heads,"Hkv":kv_heads,
                    "conversion":immediate,"before_validation":training['before']['validation'],
                    "selected_validation":training['selected_validation'],"best_adapt_epoch":training['best_epoch'],
                    "resources":resource_probe(model),"cache_check":cache_check,
                    "cached_vs_full_greedy_equal":equal,"cached_generation":cached,
                    "full_generation":recomputed,"reload_max_error":reload_error,
                    "fingerprint":fingerprint}
        history[name]=training
        print(f"{name}: Hkv={kv_heads}, 转换后val={rows[name]['before_validation']['loss']:.6f}, "
              f"所选val={rows[name]['selected_validation']['loss']:.6f}, KV={rows[name]['resources']['actual_cache_bytes']} bytes")
    if model_fingerprint(source)!=before_fingerprint:
        raise AssertionError('源 MHA 被修改了')
    report={"environment":{"python":platform.python_version(),"torch":str(torch.__version__),
                           "device":str(device),"threads":torch.get_num_threads(),"seed":seed},
            "source_validation":source_metrics,"source_unchanged":True,
            "native_sdpa":native_gqa_probe(device),"models":rows,
            "limitations":["人工模板、单种子、验证选轮次，无独立测试集；不是架构排名", 
                           "组均值转换通常有损；续训收益不保证，不是原论文5%算力复现", 
                           "不同结构参数/算力不同，只保证共同起点与相同续训更新次数", 
                           "载荷不是峰值显存；未做性能基准；不声称提速比例", 
                           "缓存仅无 padding 等长批次；模型更新后必须清空旧缓存"]}
    for name,data in [('metrics.json',report),('training.json',{'source':source_training,'adaptation':history}),
                      ('split.json',{'train':train_texts,'validation':val_texts,'vocab':vocab})]:
        (output_dir/name).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    print('结果目录：',output_dir.resolve())
    return report

## 8. 自动检查：先确认计算正确

下面自动检查覆盖：连续分组、非法头数、三种结构的输出/梯度、官方展开SDPA、非连续输入、紧凑缓存、因果性、padding、位置偏移、投影池化、特殊等价情形、保存重载，以及输入检查点兼容。

可选的原生 GQA 如果在本机不支持，会明确输出原因；不影响基础实现。通过有限测试不是“没有任何bug”的证明，新增功能时应继续增加案例。

In [24]:
class Checks:
    def __init__(self):
        self.names = []

    def close(self, name, actual, expected, **kwargs):
        torch.testing.assert_close(actual, expected, **kwargs)
        self.names.append(name)

    def true(self, name, condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def raises(self, name, fn, expected=ValueError):
        try:
            fn()
        except expected:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}：没有抛出 {expected.__name__}")

def run_checks(device: torch.device = torch.device('cpu')) -> dict:
    """检查项均显式登记；可选原生 GQA 不可用时记为 unsupported，不伪装成通过。"""
    c=Checks()
    set_seed(2201)
    dtype=torch.float64
    c.true('MHA 每个 Q 对应独立 KV',head_mapping(4,4)==[0,1,2,3])
    c.true('GQA 相邻 Q 共享 KV',head_mapping(8,2)==[0,0,0,0,1,1,1,1])
    c.true('MQA 全部 Q 共享 KV',head_mapping(4,1)==[0,0,0,0])
    c.raises('拒绝不能整除的头数',lambda:head_mapping(4,3))
    c.raises('拒绝零 KV 头',lambda:head_mapping(4,0))
    c.raises('拒绝布尔头数',lambda:head_mapping(True,1))
    c.raises('配置拒绝不整除',lambda:ModelConfig(vocab_size=10,n_kv_heads=3))
    c.raises('配置拒绝奇数旋转维度',lambda:ModelConfig(vocab_size=10,d_model=12))
    tags=torch.tensor([10.,20.],device=device).reshape(1,2,1,1)
    c.close('repeat_interleave 连续分组',repeat_kv(tags,4).flatten(),tags.new_tensor([10,10,20,20]))
    c.true('普通 repeat 的顺序不同',not torch.equal(repeat_kv(tags,4),tags.repeat(1,2,1,1)))
    c.close('Hkv=Hq 不改变数值',repeat_kv(tags,2),tags)
    leaf=tags.clone().requires_grad_()
    repeat_kv(leaf,8).sum().backward()
    c.close('共享 KV 的梯度累加',leaf.grad,torch.full_like(leaf,4))
    c.true('零长度缓存估算',kv_payload_estimate(2,1,0,2,12,4)==0)
    c.raises('拒绝负长度估算',lambda:kv_payload_estimate(2,1,-1,2,12,4))

    for hkv in (4,2,1):
        prefix=f'Hkv={hkv}'
        q=torch.randn(2,4,3,4,device=device,dtype=dtype,requires_grad=True)
        k=torch.randn(2,hkv,5,4,device=device,dtype=dtype,requires_grad=True)
        v=torch.randn(2,hkv,5,4,device=device,dtype=dtype,requires_grad=True)
        allowed=offset_causal_mask(2,3,device)
        output,weights=grouped_attention(q,k,v,allowed)
        ref,ref_weights=attention_repeat_reference(q,k,v,allowed)
        c.close(prefix+' 分组与重复输出',output,ref,atol=1e-10,rtol=1e-9)
        c.close(prefix+' 分组与重复权重',weights,ref_weights,atol=1e-10,rtol=1e-9)
        c.true(prefix+' 输出和权重保留 Hq',output.shape==(2,4,3,4) and weights.shape==(2,4,3,5))
        c.close(prefix+' 权重行和',weights.sum(-1),torch.ones(2,4,3,device=device,dtype=dtype))
        c.close(prefix+' 屏蔽位置精确为零',weights.masked_select(~allowed.expand_as(weights)),
                torch.zeros_like(weights.masked_select(~allowed.expand_as(weights))))
        # 第三条独立实现：显式逐个 Query 头选对应 KV 头。
        rows=[]
        for h,g in enumerate(head_mapping(4,hkv)):
            score=q[:,h]@k[:,g].transpose(-2,-1)/2
            a=torch.softmax(score.masked_fill(~allowed[:,0],float('-inf')),-1)
            rows.append(a@v[:,g])
        c.close(prefix+' 逐 Q 头循环参考',output,torch.stack(rows,dim=1),atol=1e-10,rtol=1e-9)
        grad1=torch.autograd.grad(output.square().sum(),(q,k,v),retain_graph=True)
        grad2=torch.autograd.grad(ref.square().sum(),(q,k,v),retain_graph=True)
        for name,a,b in zip(('Q','K','V'),grad1,grad2):
            c.close(prefix+f' {name}梯度',a,b,atol=1e-10,rtol=1e-9)
        official=F.scaled_dot_product_attention(q,repeat_kv(k,4),repeat_kv(v,4),
                                              attn_mask=allowed,dropout_p=0.0,is_causal=False)
        c.close(prefix+' 官方展开 SDPA',output,official,atol=1e-10,rtol=1e-9)
        # 构造非连续但值相同的输入。
        q_nc=q.transpose(-1,-2).contiguous().transpose(-1,-2)
        c.close(prefix+' 非连续 Query',grouped_attention(q_nc,k,v,allowed)[0],output)
        c.close(prefix+' eval 关闭 dropout',grouped_attention(q,k,v,allowed,dropout_p=0.5,training=False)[0],output)

    # 共享 KV 不代表 Q 的权重/输出相同。
    q=torch.tensor([[[[2.,0.]],[[0.,2.]]]],device=device,dtype=dtype)
    k=torch.eye(2,device=device,dtype=dtype)[None,None]
    v=k.clone()
    mask=torch.ones(1,1,1,2,device=device,dtype=torch.bool)
    o,w=grouped_attention(q,k,v,mask)
    c.true('共享 KV 时两个 Q 仍可不同',not torch.allclose(w[:,0],w[:,1]))
    c.raises('拒绝全遮挡',lambda:grouped_attention(q,k,v,torch.zeros_like(mask)))
    c.raises('拒绝浮点布尔混淆 mask',lambda:grouped_attention(q,k,v,mask.float()))
    c.raises('拒绝不匹配 KV 形状',lambda:grouped_attention(q,k,v[:,:,:1],mask))
    c.raises('拒绝不同 dtype',lambda:grouped_attention(q,k.float(),v,mask))
    c.raises('拒绝 dropout=1',lambda:grouped_attention(q,k,v,mask,dropout_p=1.0))

    weight=torch.arange(32.,device=device,dtype=dtype).reshape(8,4)
    c.close('权重池化按头而非坐标',pool_kv_projection(weight,4,2,2),
            torch.stack(((weight[:2]+weight[2:4])/2,(weight[4:6]+weight[6:8])/2)).reshape(4,4))
    bias=torch.arange(8.,device=device,dtype=dtype)
    c.close('偏置同样池化',pool_kv_projection(bias,4,2,2),bias.new_tensor([1,2,5,6]))
    c.close('不减少头时保持权重',pool_kv_projection(weight,4,4,2),weight)
    c.raises('拒绝错误投影形状',lambda:pool_kv_projection(weight,4,2,4))

    cfg=ModelConfig(vocab_size=10,d_model=16,n_layers=2,swiglu_dim=32,ff_dim=64,max_len=16)
    source=ModernGPT(cfg).to(device).eval()
    ids=torch.tensor([[BOS_ID,4,5,4,6],[BOS_ID,5,4,6,5]],device=device)
    source_fingerprint=model_fingerprint(source)
    for hkv in (4,2,1):
        prefix=f'模型 Hkv={hkv}'
        model=convert_mha(source,hkv).eval()
        c.true(prefix+' 配置保持 Q 头',model.cfg.n_heads==4 and model.cfg.n_kv_heads==hkv)
        c.true(prefix+' K 投影实际缩小',model.blocks[0].attn.k_proj.weight.shape==(hkv*4,16))
        c.true(prefix+' Q 投影不变',torch.equal(model.blocks[0].attn.q_proj.weight,source.blocks[0].attn.q_proj.weight))
        c.true(prefix+' 输出投影不变',torch.equal(model.blocks[0].attn.out_proj.weight,source.blocks[0].attn.out_proj.weight))
        c.true(prefix+' 注意力参数量公式',count_parameters(model.blocks[0].attn)==attention_parameter_count(model.cfg)['total_per_layer'])
        with torch.no_grad():
            full=model(ids)
            _,cache=model.forward_cached(ids)
            expected=kv_payload_estimate(2,2,5,hkv,4,4)
            c.true(prefix+' 每层紧凑缓存',all(pair[0].shape==(2,hkv,5,4) for pair in cache.layers))
            c.true(prefix+' 缓存字节公式',cache_payload_bytes(cache)==expected)
            old=tuple((a.clone(),b.clone()) for a,b in cache.layers)
            _,longer=model.forward_cached(ids[:,:1],cache)
            c.true(prefix+' 新增长度',longer.length==6)
            c.true(prefix+' 不改写旧缓存',all(torch.equal(a,x) and torch.equal(b,y) for (a,b),(x,y) in zip(cache.layers,old)))
            change=ids.clone();change[:,-1]=7
            c.close(prefix+' 因果性',full[:,:-1],model(change)[:,:-1],atol=2e-6,rtol=2e-5)
            padded=F.pad(ids,(0,2),value=PAD_ID)
            c.close(prefix+' 右 padding 不改变前缀',full,model(padded)[:,:5],atol=2e-6,rtol=2e-5)
            if hkv==4:
                c.close('MHA 原样转换输出相同',full,source(ids),atol=0,rtol=0)
            else:
                c.true(prefix+' 普通池化不是等价变换',(full-source(ids)).abs().max()>1e-7)
            c.raises(prefix+' 拒绝跨模型缓存',lambda:source.forward_cached(ids[:,:1],cache))
            c.raises(prefix+' 拒绝缓存 PAD',lambda:model.forward_cached(ids.new_tensor([[PAD_ID]])))
            c.raises(prefix+' 拒绝改变位置起点',lambda:model.forward_cached(ids[:,:1],cache,position_offset=1))
        chunk=compare_chunked(model,ids,[2,1,2],offset=9)
        c.true(prefix+' 完整与分块前向',chunk['max_logits_error']<2e-5)
        c.raises(prefix+' 拒绝有梯度的缓存调用',lambda:model.forward_cached(ids))
        model.train();model.zero_grad(set_to_none=True)
        logits=model(ids); loss=F.cross_entropy(logits.flatten(0,1),ids.flatten())
        loss.backward()
        c.true(prefix+' 所有参数收到有限梯度',all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))
        c.true(prefix+' 无缓存写入 state_dict',not any('cache' in name for name in model.state_dict()))
        vocab=SPECIAL_TOKENS+list('我爱学习模型')
        model.eval()
        a=generate_greedy(model,vocab,'我',use_cache=True,max_new_tokens=4)
        b=generate_greedy(model,vocab,'我',use_cache=False,max_new_tokens=4)
        c.true(prefix+' 本例贪心生成一致',a['new_ids']==b['new_ids'] and a['stop_reason']==b['stop_reason'])
    c.true('三路转换未改源参数',model_fingerprint(source)==source_fingerprint)
    c.raises('拒绝 GQA 充当 MHA 源',lambda:convert_mha(convert_mha(source,2),1))

    # 若每组原始 K/V 投影本来相同，按组池化才是这个特殊结构上的等价转换。
    tied=ModernGPT(cfg).to(device=device,dtype=dtype).eval()
    with torch.no_grad():
        for block in tied.blocks:
            for projection in (block.attn.k_proj,block.attn.v_proj):
                for tensor in (projection.weight,projection.bias):
                    view=tensor.reshape(2,2,4,*tensor.shape[1:])
                    view.copy_(view[:,:1].expand_as(view).clone())
        compressed=convert_mha(tied,2).eval()
        c.close('组内 KV 完全相同的特殊等价情形',tied(ids),compressed(ids),atol=1e-10,rtol=1e-9)

    with tempfile.TemporaryDirectory(prefix='lesson22_check_') as folder:
        path=Path(folder)/'gqa.pt'
        model=convert_mha(source,2).eval()
        vocab=SPECIAL_TOKENS+list('我爱学习模型')
        save_checkpoint(path,model,vocab,0)
        restored,read_vocab,_=load_checkpoint(path,device)
        with torch.no_grad():
            c.close('本课检查点严格重载',model(ids),restored(ids),atol=0,rtol=0)
        c.true('检查点保存 Hkv',restored.cfg.n_kv_heads==2 and read_vocab==vocab)
        old_cfg=asdict(cfg);old_cfg.pop('n_kv_heads')
        path21=Path(folder)/'lesson21.pt'
        torch.save({'family':'lesson21_rms_swiglu_v1','config':old_cfg,'vocab':vocab,
                    'state_dict':snapshot_state(source),'best_epoch':0},path21)
        imported,_,_=load_checkpoint(path21,device)
        with torch.no_grad():
            c.close('第21课结构兼容载入',source(ids),imported(ids),atol=0,rtol=0)
        c.true('加载不修改源文件',path21.exists())
    native=native_gqa_probe(device)
    if native['status']=='passed':
        c.true('当前环境原生 GQA 前向与梯度',True)
    else:
        print('可选原生 GQA 未支持，已记录：',native['reason'])
    print(f'通过固定检查：{len(c.names)} 项')
    return {'passed':len(c.names),'names':c.names,'native_gqa':native}

In [25]:
CHECK_REPORT=run_checks(DEVICE)
print("本机原生GQA:",CHECK_REPORT["native_gqa"])

通过固定检查：128 项
本机原生GQA: {'status': 'passed', 'device': 'cpu', 'dtype': 'float64', 'max_output_error': 3.3306690738754696e-16, 'max_gradient_error': 9.43689570931383e-16, 'note': '只验证当前环境前向和梯度；未断言 FlashAttention 或显存收益'}


## 9. 完整实验与结果阅读

下个单元执行训练及保存。重新执行会用相同种子重新开始，并覆盖同名输出；不会在上一次运行的权重上偷偷累积训练。

已有第21课检查点且希望跳过源训练，回到环境单元设置 `SOURCE_CHECKPOINT`。`ADAPT_EPOCHS=0` 则只做转换后的实验。

不应预期 MQA/GQA 一定比 MHA 更好或更差。本课数据和模型很小；观察结构修改、适应训练和缓存大小的因果关系即可。

In [26]:
REPORT=run_experiment(DEVICE,OUTPUT_DIR,epochs=EPOCHS,adapt_epochs=ADAPT_EPOCHS,
                      seed=SEED,checkpoint=SOURCE_CHECKPOINT)

mha_source    epoch=01 train=2.8283 val=2.8618


mha_source    epoch=10 train=0.4508 val=0.4804


mha_source    epoch=20 train=0.4014 val=0.4343


mha_continued epoch=01 train=0.4024 val=0.4404


mha_continued epoch=10 train=0.3895 val=0.4405
mha_continued: Hkv=4, 转换后val=0.430258, 所选val=0.427259, KV=24576 bytes


gqa           epoch=01 train=0.4028 val=0.4347


gqa           epoch=10 train=0.3915 val=0.4251
gqa: Hkv=2, 转换后val=0.441725, 所选val=0.421670, KV=12288 bytes


mqa           epoch=01 train=0.4340 val=0.4456


mqa           epoch=10 train=0.3928 val=0.4213
mqa: Hkv=1, 转换后val=0.494368, 所选val=0.419738, KV=6144 bytes
结果目录： /mnt/data/qa22/notebook_run/lesson_22_outputs


In [27]:
print("结构          参数量    转换后val   所选val     KV KiB  最佳续训轮")
for name,row in REPORT["models"].items():
    print(f"{name:13s} {row['resources']['parameters']:6d}  "
          f"{row['before_validation']['loss']:.6f}   {row['selected_validation']['loss']:.6f}  "
          f"{row['resources']['actual_cache_bytes']/1024:6.1f}     {row['best_adapt_epoch']}")
print("源模型参数未变化:",REPORT["source_unchanged"])
print("本次没有独立测试集，也没有速度/峰值显存基准。")

结构          参数量    转换后val   所选val     KV KiB  最佳续训轮
mha_continued  59856  0.430258   0.427259    24.0     4
gqa            55152  0.441725   0.421670    12.0     8
mqa            52800  0.494368   0.419738     6.0     9
源模型参数未变化: True
本次没有独立测试集，也没有速度/峰值显存基准。


### 9.1 跨架构变化与缓存数值误差不是同一个指标

`conversion.max_logits_change`：同一个输入，从源MHA变成新结构后改变了多少。通常不为零，不要求接近零。

`cache_check.max_logits_error`：对**同一个已确定模型**，整段前向与分块缓存的差异。应在浮点容差内很小。

不能把“转换造成的数值变化”当成缓存实现错误，也不能因为缓存等价就宣布模型转换无损。

In [28]:
for name,row in REPORT["models"].items():
    print(name)
    print("  直接转换的最大logits变化:",row["conversion"]["max_logits_change"])
    print("  同模型缓存的最大logits误差:",row["cache_check"]["max_logits_error"])
    print("  持久K形状:",row["resources"]["cache_shape_per_tensor"])
    print("  缓存/重算贪心一致:",row["cached_vs_full_greedy_equal"])
    print("  生成:",row["cached_generation"]["text"])

mha_continued
  直接转换的最大logits变化: 0.0
  同模型缓存的最大logits误差: 9.5367431640625e-07
  持久K形状: [1, 4, 32, 12]
  缓存/重算贪心一致: True
  生成: 我今天正在整理笔记。
gqa
  直接转换的最大logits变化: 0.42761126160621643
  同模型缓存的最大logits误差: 9.5367431640625e-07
  持久K形状: [1, 2, 32, 12]
  缓存/重算贪心一致: True
  生成: 我今天正在整理笔记。
mqa
  直接转换的最大logits变化: 0.7955981492996216
  同模型缓存的最大logits误差: 1.430511474609375e-06
  持久K形状: [1, 1, 32, 12]
  缓存/重算贪心一致: True
  生成: 我今天正在整理笔记。


### 9.2 再看一次紧凑缓存真的保存了什么

本次载荷核对固定长度为32，避免某个模型早遇EOS，缓存自然更短造成混淆。它测量的是张量有效元素字节数，不是GPU分配峰值。

下个单元加载本课 GQA 检查点，先处理前3个位置，再处理剩余2个位置。注意历史 K 已旋转；旧缓存不应变长或被改写，新缓存独立返回。

In [29]:
cached_model,cached_vocab,_=load_checkpoint(OUTPUT_DIR/"gqa.pt",DEVICE)
ids=torch.tensor([[BOS_ID,4,5,4,6]],device=DEVICE)
with torch.no_grad():
    _,old_cache=cached_model.forward_cached(ids[:,:3],position_offset=10)
    _,new_cache=cached_model.forward_cached(ids[:,3:],old_cache,position_offset=10)
print("旧长度:",old_cache.length,"新长度:",new_cache.length)
print("旧K形状:",old_cache.layers[0][0].shape)
print("新K形状:",new_cache.layers[0][0].shape)
assert new_cache.layers[0][0].shape[1]==cached_model.cfg.n_kv_heads

旧长度: 3 新长度: 5
旧K形状: torch.Size([1, 2, 3, 12])
新K形状: torch.Size([1, 2, 5, 12])


## 10. 独立练习（在这里停下来）

三个开关默认 `False`，所以直接运行全部单元不会误把“尚未完成”当成完成。把函数补全，再只打开对应开关。

**A：实现连续分组的 KV 展开。** 输入 `[B,Hkv,S,d]`，返回 `[B,Hq,S,d]`；保持梯度，不使用detach。

**B：实现分组注意力核心。** 不调用本课 `grouped_attention` 或 `attention_repeat_reference`，不使用SDPA，不重复K/V；支持矩形mask。返回输出与权重。

**C：实现投影池化。** 支持权重矩阵和偏置向量；检查头数；对组内头求平均；不要修改输入。

测试函数给出精确目标，参考答案集中放在后面。

In [30]:
def verify_exercise_a(fn):
    x=torch.tensor([10.,20.],dtype=torch.float64).reshape(1,2,1,1).requires_grad_()
    before=x.detach().clone()
    y=fn(x,4)
    torch.testing.assert_close(y.flatten(),torch.tensor([10.,10.,20.,20.],dtype=x.dtype))
    torch.testing.assert_close(fn(x,2),x)
    y.sum().backward()
    torch.testing.assert_close(x.grad,torch.full_like(x,2))
    torch.testing.assert_close(x.detach(),before)
    z=torch.arange(48.,dtype=torch.float64).reshape(2,2,3,4).transpose(-1,-2)
    torch.testing.assert_close(fn(z,8),z[:,[0,0,0,0,1,1,1,1]])
    try:
        fn(x,3)
    except ValueError:
        pass
    else:
        raise AssertionError('必须拒绝不能整除的头数')
    print('练习A通过：映射、恒等边界、梯度、非连续输入与非法头数。')


def verify_exercise_b(fn):
    gen=torch.Generator().manual_seed(2230)
    for hkv in (4,2,1):
        q=torch.randn(2,4,2,4,dtype=torch.float64,generator=gen,requires_grad=True)
        k=torch.randn(2,hkv,4,4,dtype=torch.float64,generator=gen,requires_grad=True)
        v=torch.randn(2,hkv,4,4,dtype=torch.float64,generator=gen,requires_grad=True)
        mask=offset_causal_mask(2,2)
        actual,weights=fn(q,k,v,mask)
        expected,ref_weights=attention_repeat_reference(q,k,v,mask)
        torch.testing.assert_close(actual,expected,atol=1e-10,rtol=1e-9)
        torch.testing.assert_close(weights,ref_weights,atol=1e-10,rtol=1e-9)
        a=torch.autograd.grad(actual.square().sum(),(q,k,v),retain_graph=True)
        b=torch.autograd.grad(expected.square().sum(),(q,k,v))
        for left,right in zip(a,b):
            torch.testing.assert_close(left,right,atol=1e-10,rtol=1e-9)
    # 零Query：每个允许位置的概率相等，末位置应该等于V的均值。
    q=torch.zeros(1,2,1,2,dtype=torch.float64)
    k=torch.zeros(1,1,3,2,dtype=torch.float64)
    v=torch.tensor([[[[1.,2.],[3.,4.],[5.,6.]]]],dtype=torch.float64)
    result,_=fn(q,k,v,torch.ones(1,1,1,3,dtype=torch.bool))
    torch.testing.assert_close(result,torch.tensor([[[[3.,4.]],[[3.,4.]]]],dtype=torch.float64))
    print('练习B通过：三种头数、矩形mask、前向、Q/K/V梯度与手算。')


def verify_exercise_c(fn):
    tensor=torch.arange(32.,dtype=torch.float64).reshape(8,4).requires_grad_()
    before=tensor.detach().clone()
    result=fn(tensor,4,2,2)
    expected=torch.cat(((tensor[:2]+tensor[2:4])/2,(tensor[4:6]+tensor[6:8])/2),dim=0)
    torch.testing.assert_close(result,expected)
    result.sum().backward()
    torch.testing.assert_close(tensor.grad,torch.full_like(tensor,0.5))
    torch.testing.assert_close(tensor.detach(),before)
    bias=torch.arange(8.,dtype=torch.float64)
    torch.testing.assert_close(fn(bias,4,2,2),torch.tensor([1.,2.,5.,6.],dtype=torch.float64))
    torch.testing.assert_close(fn(tensor,4,4,2),tensor)
    torch.testing.assert_close(fn(tensor,4,1,2),tensor.reshape(4,2,4).mean(0))
    try:
        fn(tensor,4,3,2)
    except ValueError:
        pass
    else:
        raise AssertionError('必须拒绝不能整除的分组')
    print('练习C通过：权重、偏置、MHA/MQA端点、梯度和不修改输入。')

In [31]:
def repeat_kv_ex(x, n_query_heads):
    raise NotImplementedError("请实现连续分组的KV展开")

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    verify_exercise_a(repeat_kv_ex)
else:
    print("练习A尚未运行；补全后打开RUN_EXERCISE_A。")

练习A尚未运行；补全后打开RUN_EXERCISE_A。


In [32]:
def grouped_attention_ex(q,k,v,allowed):
    raise NotImplementedError("请实现分组注意力，返回(output,weights)")

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    verify_exercise_b(grouped_attention_ex)
else:
    print("练习B尚未运行；补全后打开RUN_EXERCISE_B。")

练习B尚未运行；补全后打开RUN_EXERCISE_B。


In [33]:
def pool_projection_ex(tensor,n_query_heads,n_kv_heads,head_dim):
    raise NotImplementedError("请实现对KV投影按组均值池化")

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    verify_exercise_c(pool_projection_ex)
else:
    print("练习C尚未运行；补全后打开RUN_EXERCISE_C。")

练习C尚未运行；补全后打开RUN_EXERCISE_C。


## 11. 参考答案：先独立完成，再展开核对

答案使用不同函数名，不覆盖你的练习。A用增加组维再展开，B用einsum表达计算，C显式逐组平均，避免三个答案都只是调用现成函数。

检查的是你实现的功能，而不是你是否使用某一种缩写。

In [34]:
def repeat_kv_solution(x,n_query_heads):
    if not isinstance(x,torch.Tensor) or x.ndim!=4 or min(x.shape)<=0:
        raise ValueError('需要非空[B,Hkv,S,d]张量')
    head_mapping(n_query_heads,x.shape[1])
    b,hkv,s,d=x.shape
    group=n_query_heads//hkv
    # 插入组内Query头轴后展开；reshape通常需要新的存储，不把它保存到缓存。
    return x.unsqueeze(2).expand(b,hkv,group,s,d).reshape(b,n_query_heads,s,d)

verify_exercise_a(repeat_kv_solution)

练习A通过：映射、恒等边界、梯度、非连续输入与非法头数。


In [35]:
def grouped_attention_solution(q,k,v,allowed):
    _attention_inputs(q,k,v,allowed)
    b,hq,t,d=q.shape
    hkv,s=k.shape[1],k.shape[-2]
    queries=q.reshape(b,hkv,hq//hkv,t,d)
    # g是KV组，r是组内的Query头；在d轴求内积。
    scores=torch.einsum('bgrtd,bgsd->bgrts',queries,k)/math.sqrt(d)
    weights=torch.softmax(scores.masked_fill(~allowed.unsqueeze(2),float('-inf')),dim=-1)
    heads=torch.einsum('bgrts,bgsd->bgrtd',weights,v)
    return heads.reshape(b,hq,t,d),weights.reshape(b,hq,t,s)

verify_exercise_b(grouped_attention_solution)

练习B通过：三种头数、矩形mask、前向、Q/K/V梯度与手算。


In [36]:
def pool_projection_solution(tensor,n_query_heads,n_kv_heads,head_dim):
    head_mapping(n_query_heads,n_kv_heads)
    positive_int(head_dim,'head_dim')
    if (not isinstance(tensor,torch.Tensor) or tensor.ndim not in (1,2)
            or not tensor.is_floating_point() or tensor.shape[0]!=n_query_heads*head_dim):
        raise ValueError('投影权重或偏置的形状不匹配')
    per_head=tensor.reshape(n_query_heads,head_dim,*tensor.shape[1:])
    group=n_query_heads//n_kv_heads
    result=[]
    for g in range(n_kv_heads):
        result.append(per_head[g*group:(g+1)*group].mean(dim=0))
    return torch.stack(result).reshape(n_kv_heads*head_dim,*tensor.shape[1:])

verify_exercise_c(pool_projection_solution)

练习C通过：权重、偏置、MHA/MQA端点、梯度和不修改输入。


## 12. 验收与常见错误

能够不看代码回答：

- `Hq=8,Hkv=2`，每个Query对应哪个KV？为什么直接 `repeat` 不对？
- 为什么MQA不是单头注意力？共享KV为什么仍能有不同的注意力权重？
- `Hkv` 降为四分之一，哪些内存按比例降低，哪些没有？
- 缓存里存的是旋转前还是旋转后的K？有没有重复头？
- 为什么平均合并MHA投影通常不能保持模型输出？为什么续训MHA也要作为对照？

常见错误：把每头维度算成 `D/Hkv`；按特征坐标而不是按头池化；重复缓存到Hq个头；取错分组顺序；共享KV时切断梯度；把所有Query头也平均；修改模型后继续使用旧缓存。

`NaN` 先检查是否整行mask都禁止、输入是否有限；形状错误先打印 `Q/K/V` 和缓存的各条轴。有限测试通过后，仍需为新输入和新功能补充测试。

下一课：**SDPA、FlashAttention 与推理性能测量**。把清晰的教学实现与实际内核区分开，分别测正确性、延迟和峰值显存。

## 参考资料与本课边界

[1] Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/html/2305.13245v3)。用于定义分组、按组均值池化和额外训练方法；不把原论文质量/速度结果移植到本课。

[2] Shazeer, [Fast Transformer Decoding: One Write-Head is All You Need](https://arxiv.org/abs/1911.02150)。用于MQA、KV共享与带宽动机。

[3] PyTorch, [scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)。用于GQA参数、头数约束、mask与后端说明。具体支持以本机探测为准。

[4] Meta, [Llama 3公开实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)。核对独立 `n_heads/n_kv_heads`、投影宽度、紧凑缓存与重复读取的顺序；本课不是其配置复制品。

[5] Hugging Face, [How caching works](https://huggingface.co/docs/transformers/cache_explanation)。用于请求级缓存与prefill/decode基础。

**验证环境说明：** 随文件保留的是本次CPU执行输出。用户设备、版本和随机行为可能影响浮点细节与训练结果。本课没有实测RTX 3060，也没有报告GPU性能数字。缓存仅支持无padding等长批次；没有量化、分页缓存、张量并行、滑动窗口或服务调度。

完整脚本用 `python lesson_22_gqa_mqa_and_compact_kv_cache.py --checks-only` 检查，用无参数命令运行完整实验。Notebook无需导入或运行该脚本。